# FastAPI 基础演示

本 notebook 演示 FastAPI 的核心用法，包括：
- 创建应用
- 路由与请求方法
- 路径参数、查询参数
- Pydantic 请求体
- 启动服务并测试

## 1. 安装依赖

确保已安装 fastapi 和 uvicorn。

In [ ]:
# 安装依赖（如已安装可跳过）
!uv add fastapi uvicorn

## 2. 创建一个最小 FastAPI 应用

定义 GET 和 POST 路由。

In [1]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="FastAPI Demo")


class Item(BaseModel):
    name: str
    price: float
    is_offer: bool | None = None


@app.get("/")
def read_root():
    return {"message": "Hello FastAPI"}


@app.get("/items/{item_id}")
def read_item(item_id: int, q: str | None = None):
    return {"item_id": item_id, "q": q}


@app.post("/items")
def create_item(item: Item):
    return {"item_name": item.name, "item_price": item.price}


print("FastAPI 应用已创建")

FastAPI 应用已创建


## 3. 使用 TestClient 测试路由

无需启动服务器，直接用 `TestClient` 测试。

In [2]:
from fastapi.testclient import TestClient

client = TestClient(app)

# 测试 GET /
response = client.get("/")
print("GET /:", response.status_code, response.json())

# 测试 GET /items/42?q=test
response = client.get("/items/42?q=test")
print("GET /items/42:", response.status_code, response.json())

# 测试 POST /items
response = client.post("/items", json={"name": "Apple", "price": 5.5})
print("POST /items:", response.status_code, response.json())

GET /: 200 {'message': 'Hello FastAPI'}
GET /items/42: 200 {'item_id': 42, 'q': 'test'}
POST /items: 200 {'item_name': 'Apple', 'item_price': 5.5}


/root/rag-agent-learning/.venv/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 4. 与 RAG 流程集成

调用项目里已有的 `rag.pipeline.ask` 实现问答接口。

In [3]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from rag.pipeline import ask

rag_app = FastAPI(title="RAG Q&A API")


class AskRequest(BaseModel):
    question: str


@rag_app.post("/ask")
def ask_endpoint(request: AskRequest):
    try:
        result = ask(request.question)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc

    return {
        "question": result["input"],
        "answer": result["answer"],
        "contexts": [doc.page_content for doc in result["context"]],
    }


rag_client = TestClient(rag_app)

# 注意：运行前需要确保向量库已构建
response = rag_client.post("/ask", json={"question": "小说主角是谁？"})
print("RAG /ask:", response.status_code, response.json())

ModuleNotFoundError: No module named 'rag'

## 5. 启动真实服务器（可选）

在终端中运行：

```bash
uv run uvicorn api:app --reload --host 0.0.0.0 --port 8000
```

然后用 curl 或浏览器访问：

```bash
curl -X POST "http://localhost:8000/ask" -H "Content-Type: application/json" -d '{"question":"小说主角是谁？"}'
```